# Jour 2 · Convolution et premier CNN

> Version corrigée — les activités et leurs explications sont remplies.

## Objectifs

- comprendre filtre, convolution, carte de caractéristiques et pooling
- entraîner un petit CNN sur les mêmes images
- expliquer pourquoi les CNN conviennent mieux aux structures spatiales


## Le problème du réseau Dense

En aplatissant l'image, nous traitons les pixels comme une longue liste. Un **CNN**, ou réseau convolutif, conserve la disposition spatiale et cherche de petits motifs locaux : bords, orientations, textures puis formes plus complexes.


## Un filtre est une petite grille de poids

Un **filtre**, ou *kernel*, reçoit une petite zone de l'image. On multiplie les pixels et les poids case par case, puis on additionne les produits et un biais. Le filtre est ensuite déplacé pour refaire ce calcul ailleurs. En Deep Learning, on appelle couramment cette opération une **convolution** ; les logiciels appliquent en pratique le filtre sans le retourner.

![Zone trois par trois multipliée par un filtre puis somme des produits](../../ressources/illustrations/jour_02/03_convolution_calcul.svg)

La **carte de caractéristiques**, ou *feature map*, range toutes les réponses d'un filtre aux différentes positions. Une réponse forte indique que cette zone active le filtre ; sa signification est apprise par l'entraînement. Le filtre du dessin sert seulement à montrer le calcul ; dans le modèle, ses poids seront appris.

Le **partage des poids** signifie que le même filtre est appliqué partout. On apprend ainsi un détecteur local, au lieu d'apprendre une version différente à chaque position.


## À vous de jouer — calculer une réponse de filtre

La somme du schéma vaut 2 quand le poids central vaut −1 et le biais 0. Sans coder la convolution entière, remplacez ce poids central par 0 : de combien la réponse augmente-t-elle ? Complétez le nombre attendu.


In [ ]:
reponse_filtre = 4  # Quatre produits valent 1 ; les autres valent 0.
print("Nouvelle réponse :", reponse_filtre, "; augmentation :", reponse_filtre - 2)


### Lecture du corrigé

Le pixel central vaut 2 : passer son poids de −1 à 0 ajoute 2 à la somme. La réponse passe de 2 à 4. Le poids détermine donc la contribution de ce pixel ; la convolution combine les contributions locales.


## Le pooling résume les réponses voisines

Le **max pooling** découpe une carte en petites zones et conserve le maximum de chaque zone. Une zone 2 × 2 de quatre valeurs devient une seule valeur. Il réduit la largeur et la hauteur, sans apprendre de poids et sans changer le nombre de canaux.

![Pooling numérique de quatre zones deux par deux vers une matrice deux par deux](../../ressources/illustrations/jour_02/03_maxpool_calcul.svg)

Le dessin montre le principe sur une carte 4 × 4 ; le modèle fera la même chose sur 8 × 8. Une réponse forte peut rester visible après cette réduction, mais l'emplacement exact et certaines valeurs sont perdus.

Petit contrôle : quel maximum gardez-vous pour la zone en bas à droite ? Pourquoi une matrice 8 × 8 devient-elle 4 × 4 ?


## Suivre les dimensions d'un premier CNN

![Images et cartes empilées montrant les formes avant et après convolution et pooling](../../ressources/illustrations/jour_02/03_dimensions_simple.svg)

`Conv2D(16, 3)` apprend 16 filtres de taille 3 × 3. **Padding** désigne la bordure ajoutée pour effectuer les calculs près des bords. Avec `padding="same"` et un déplacement d'une case, la hauteur et la largeur restent 8 × 8.

La convolution produit 16 cartes ; ReLU remplace leurs valeurs négatives par zéro. `MaxPooling2D(2)` réduit chacune à 4 × 4. `Flatten` range alors `4 × 4 × 16 = 256` valeurs dans une liste. Une couche Dense les transforme en dix scores softmax.

Nous utilisons d'abord **une seule convolution**, puis pooling et classification. Une deuxième convolution peut combiner des motifs plus tard, après compréhension de cette première chaîne.


## Préparer les outils

Un **import** rend un outil Python disponible. `as np`, `as plt` et `as tf` lui donnent un nom court. NumPy gère les tableaux ; Matplotlib les affiche ; TensorFlow entraîne les réseaux. Les premières lignes configurent les messages et le calcul sur CPU. Elles se recopient telles quelles et ne font pas partie du modèle.

La **graine aléatoire** fixe le point de départ du hasard. `42` est un choix arbitraire : il facilite les comparaisons sans garantir des résultats exactement identiques sur toutes les machines.


In [ ]:
import os  # Réglages de l'environnement avant le chargement de TensorFlow.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")  # Réduit les messages techniques.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")  # Utilise le CPU pour ce petit cours.

import numpy as np  # Tableaux et calculs numériques.
import matplotlib.pyplot as plt  # Graphiques et images.
import tensorflow as tf  # Réseaux et calcul automatique des gradients.

tf.keras.utils.set_random_seed(42)  # Fixe le hasard pour comparer les expériences.
try:
    # Limite le nombre de threads utilisés par TensorFlow.
    # Un thread par opération suffit ; ce réglage limite la consommation du cours.
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass  # Ce réglage ne peut plus changer si TensorFlow a déjà commencé à calculer.
print("TensorFlow :", tf.__version__)


## Reprendre les mêmes images, avec trois rôles distincts

`load_digits()` fournit de petites images de chiffres. `astype("float32")` convertit les pixels en nombres décimaux ; `int64` garde les labels sous forme d'entiers. La division par `16` place les pixels entre 0 et 1. Ce maximum vient du jeu de données, pas d'une valeur estimée sur le test.

**Train** : exemples utilisés pour corriger les poids. **Validation** : exemples utilisés pour suivre l'entraînement et choisir les réglages. **Test** : exemples réservés à la mesure finale. `train_test_split` coupe ensemble les images et leurs labels, pour conserver leur association. `stratify=labels` conserve approximativement la proportion de chaque chiffre. `random_state=42` fixe ce découpage.

Dans les noms Python, `X` désigne les entrées et `y` les réponses. Nous séparons d'abord 25 % pour le test, puis 20 % du reste pour la validation. Les proportions finales sont environ 60 % train, 15 % validation et 25 % test.


In [ ]:
from sklearn.datasets import load_digits  # Données incluses localement dans scikit-learn.
from sklearn.model_selection import train_test_split  # Découpage sans mélanger images et labels.

digits = load_digits()
images = digits.images.astype("float32") / 16.0  # Intensités de 0 à 1.
labels = digits.target.astype("int64")  # Chiffre attendu, de 0 à 9.
# Les pixels vont à l'origine de 0 à 16 ; on les ramène entre 0 et 1 pour faciliter l'apprentissage.
# stratify garde à peu près la même proportion de chaque chiffre dans les jeux.
# random_state rend le découpage reproductible.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    images, labels, test_size=0.25, stratify=labels, random_state=42,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, stratify=y_train_val, random_state=42,
)
print("Train :", X_train.shape, "Validation :", X_val.shape, "Test :", X_test.shape)
print("Pixels du train :", X_train.min(), "à", X_train.max())


## Ajouter le canal avant d'utiliser Conv2D

Les images sont actuellement rangées en `(nombre, 8, 8)`. `Conv2D` attend `(nombre, hauteur, largeur, canaux)`. Une image en niveaux de gris possède un seul canal : nous ajoutons un axe de taille `1`, sans modifier les pixels.

Dans `X_train[..., np.newaxis]`, `...` conserve tous les axes présents et `np.newaxis` ajoute un nouvel axe à la fin. La forme devient `(nombre, 8, 8, 1)`. Le dernier `1` est le nombre de canaux ; le premier nombre reste le nombre d'images.


In [ ]:
X_train_cnn = X_train[..., np.newaxis]  # Ajoute le canal unique de chaque image.
X_val_cnn = X_val[..., np.newaxis]  # La validation doit avoir exactement le même format.
X_test_cnn = X_test[..., np.newaxis]  # Même transformation pour le test.
print("Avant :", X_train.shape, "Après :", X_train_cnn.shape)
assert X_train_cnn.shape[-1] == 1  # Contrôle : un seul canal de gris. Les pixels sont inchangés.


In [ ]:
cnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(8, 8, 1)),  # Forme d'une image, sans l'axe du batch.
    tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu", name="conv_1"),
    # Les 16 filtres cherchent des motifs locaux ; la hauteur et la largeur restent 8.
    tf.keras.layers.MaxPooling2D(2),  # Maximum de chaque zone 2 × 2 : 8 × 8 devient 4 × 4.
    tf.keras.layers.Flatten(name="features"),  # Réorganise 4 × 4 × 16 en 256 nombres.
    tf.keras.layers.Dense(10, activation="softmax"),  # Un score par chiffre, somme égale à 1.
])
cnn.summary()  # Affiche les formes et le nombre de valeurs apprises par couche.


## Choisir la loss avant de lancer l'entraînement

Le réseau donne dix scores ; la bonne réponse est un entier comme `7`. `sparse_categorical_crossentropy` compare les deux : elle pénalise un score faible pour la bonne classe. **Sparse** signifie ici que le label est un entier, au lieu d'un tableau de dix zéros avec un `1` à la bonne position. **Categorical** indique plusieurs classes exclusives : une image reçoit un seul chiffre.

La **loss** guide l'apprentissage. L'**accuracy** compte seulement la proportion de classes correctement choisies. Une prédiction correcte mais hésitante peut donc avoir la même accuracy qu'une prédiction correcte et nette, tout en ayant une loss différente.

`compile` associe le réseau, la loss et l'optimiseur. `optimizer="adam"` demande à Adam de modifier les poids grâce aux gradients. `metrics=["accuracy"]` ajoute une mesure à afficher ; elle ne remplace pas la loss.


In [ ]:
cnn.compile(
    optimizer="adam",  # Optimiseur des poids de convolution et des couches Dense.
    loss="sparse_categorical_crossentropy",  # Dix classes, avec un entier par réponse.
    metrics=["accuracy"],  # Mesure supplémentaire lisible.
)


## Apprendre, puis suivre la validation

`fit` lance l'entraînement. Une **époque** est un parcours du train ; `batch_size=32` traite 32 images avant une correction des poids. Les images de `validation_data` sont utilisées pour mesurer `val_loss` et `val_accuracy`, jamais pour cette correction.

Un **callback** est une action appelée pendant l'entraînement. `EarlyStopping` observe `val_loss` : après plusieurs époques sans amélioration, il arrête le calcul. `restore_best_weights=True` recharge les meilleurs poids observés sur la validation. `epochs` fixe seulement un maximum ; `verbose=0` cache le journal détaillé.


In [ ]:
arret_anticipe = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True,
)  # Conserve la meilleure époque de validation.
history = cnn.fit(
    X_train_cnn, y_train,  # Apprentissage sur les images avec leur canal.
    validation_data=(X_val_cnn, y_val),  # Contrôle sans modifier les poids.
    epochs=20, batch_size=32, callbacks=[arret_anticipe], verbose=0,
)
print("Époques effectuées :", len(history.history["loss"]))


## Lire la progression puis mesurer le résultat final

Les courbes comparent les réponses correctes du train et de la validation. Une différence durable invite à examiner la généralisation, pas seulement à ajouter des couches. `evaluate` mesure ensuite le CNN sur le test, sans apprentissage. La comparaison avec le Dense doit tenir compte du même découpage et du coût de calcul, pas seulement du score.


In [ ]:
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.xlabel("époque")
plt.ylabel("proportion de réponses correctes")
plt.title("Accuracy du CNN")
plt.legend()
plt.show()
loss, accuracy = cnn.evaluate(X_test_cnn, y_test, verbose=0)  # Mesure finale uniquement.
print("Accuracy test du CNN :", round(accuracy, 3))


## Observer une carte avant de choisir la classe

Une **sortie intermédiaire** est la sortie d'une couche à l'intérieur du réseau. `get_layer("conv_1")` retrouve la couche nommée ainsi. `tf.keras.Model(inputs=..., outputs=...)` construit une vue qui s'arrête à cette couche, sans nouvel entraînement. Cela permet de vérifier les dimensions et d'afficher ses réponses.


## À vous de jouer — lire une sortie intermédiaire

Avant d'exécuter, prévoyez la forme de sortie pour quatre images : nombre d'images, hauteur, largeur et nombre de filtres. Complétez la couche de sortie dans la vue fournie. Indice : `cnn.get_layer("conv_1").output`.


In [ ]:
sortie_choisie = cnn.get_layer("conv_1").output  # Sortie après les 16 filtres et ReLU.
modele_feature_maps = tf.keras.Model(inputs=cnn.inputs, outputs=sortie_choisie)
feature_maps = modele_feature_maps.predict(X_test_cnn[:4], verbose=0)  # Quatre images.
print("Entrée :", X_test_cnn[:4].shape, "Sortie :", feature_maps.shape)


### Lecture du corrigé

La sortie vaut `(4, 8, 8, 16)` : quatre images, huit lignes, huit colonnes et seize cartes. Les filtres sont les mêmes pour les quatre images, mais leurs réponses dépendent des pixels. Le canal d'entrée ne se confond pas avec le nombre de cartes apprises.


### Relier une forme à une image réelle

`feature_maps[0, :, :, canal]` sélectionne la première image, toutes ses lignes et colonnes, puis une carte. Les `:` signifient « garder toutes les valeurs de cet axe ». L'affichage suivant apparaît quand l'activité précédente est complétée.


In [ ]:
if feature_maps is not None:
    fig, axes = plt.subplots(1, 5, figsize=(10, 2.3))
    axes[0].imshow(X_test[0], cmap="gray_r")  # Image originale avant le réseau.
    axes[0].set_title("image")
    for canal, ax in enumerate(axes[1:]):  # enumerate fournit 0, 1, 2, 3 et leur axe graphique.
        ax.imshow(feature_maps[0, :, :, canal], cmap="viridis")  # Une carte de réponses.
        ax.set_title(f"carte {canal}")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## Ce que ce petit CNN apprend

Le réseau reçoit un chiffre déjà cadré et choisit parmi dix classes. Une bonne accuracy sur le test concerne des images construites de la même manière. Une photo couleur, une autre résolution ou un cadrage très différent peuvent changer la qualité : c'est un **décalage de distribution**.

Le partage des poids permet de chercher un motif à plusieurs endroits avec les mêmes paramètres. Des convolutions supplémentaires peuvent combiner ces réponses, mais un réseau plus grand augmente le coût et peut mémoriser davantage le train.

Les sections qui suivent sont **facultatives**. Le parcours principal est terminé lorsque vous pouvez expliquer le filtre, le pooling et chaque dimension du CNN.


## Extension facultative 1 — ajouter une deuxième convolution

![Plusieurs couches peuvent combiner les motifs locaux en motifs plus riches](../../ressources/illustrations/jour_02/03_hierarchie_motifs.png)

Une nouvelle convolution après le pooling recevrait les seize cartes, et non les pixels bruts. Avec 32 filtres et `padding="same"`, elle produirait `(4, 4, 32)`. Cela peut enrichir la représentation. Il faut le vérifier sur la validation, avec une seule modification à la fois.


## Extension facultative 2 — varier des images plausibles

L'**augmentation de données** crée des variantes pendant l'entraînement, en conservant le label. Une petite translation d'un chiffre peut rester plausible ; une rotation complète peut changer son sens. Une transformation doit représenter une variation attendue dans l'application.

Le code ci-dessous est désactivé par défaut. Avec `executer_augmentation = True`, il montre une image avant et après une petite rotation. `RandomRotation(0.04)` autorise environ 4 % d'un tour dans chaque sens. Le paramètre `training=True` demande explicitement une transformation de train.

Cette observation ne change pas le CNN entraîné. Pour tester l'effet sur l'apprentissage, il faudrait construire un autre modèle et comparer sa validation.


In [ ]:
executer_augmentation = False  # Facultatif : passer à True après avoir compris le CNN simple.
if executer_augmentation:
    augmentation = tf.keras.layers.RandomRotation(0.04)  # Rotation modérée, aléatoire.
    image_variee = augmentation(X_train_cnn[:1], training=True).numpy()  # .numpy() donne un tableau.
    fig, axes = plt.subplots(1, 2, figsize=(5, 2.4))
    axes[0].imshow(X_train[0], cmap="gray_r")
    axes[1].imshow(image_variee[0, :, :, 0], cmap="gray_r")  # Première image, canal gris.
    axes[0].set_title("originale")
    axes[1].set_title("petite rotation")
    for ax in axes:
        ax.axis("off")
    plt.show()


## Extension facultative 3 — réutiliser un extracteur déjà entraîné

![Les couches déjà entraînées sont conservées et seule la nouvelle tête apprend](../../ressources/illustrations/jour_02/03_transfert_apprentissage.png)

Le **transfert d'apprentissage** réutilise des couches qui ont déjà appris. L'**extracteur** produit ici les 256 valeurs avant la couche de classification. `trainable=False` le **gèle** : ses poids ne sont plus corrigés. Une nouvelle **tête** de sortie apprend une autre réponse : pair ou impair.

`y % 2` donne le reste de la division par deux : 0 pour un chiffre pair, 1 pour un impair. Nous codons ici « pair » par 1 et « impair » par 0. **Sigmoid** transforme la sortie d'un seul neurone en une valeur entre 0 et 1 : elle représente ici le score de la classe « pair ».

`binary_crossentropy` est la loss pour cette cible binaire : elle pénalise un score élevé pour une cible 0 et un score faible pour une cible 1. Avec un score supérieur ou égal à 0,5, on prédit « pair ». Cette sortie diffère des dix scores softmax du problème initial.

Activez `executer_transfert` pour cette expérience. Dans un vrai transfert, les couches sources viennent souvent d'un beaucoup plus grand jeu. Dégeler certaines couches et les ajuster doucement s'appelle **fine-tuning** ; cette étape est seulement située ici.


In [ ]:
executer_transfert = False  # Facultatif : activer après le parcours principal.
if executer_transfert:
    extracteur = tf.keras.Model(inputs=cnn.inputs, outputs=cnn.get_layer("features").output)
    extracteur.trainable = False  # Ses paramètres restent fixes.
    y_train_parite = (y_train % 2 == 0).astype("float32")  # 1 signifie pair, 0 impair.
    y_val_parite = (y_val % 2 == 0).astype("float32")
    y_test_parite = (y_test % 2 == 0).astype("float32")
    modele_parite = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(8, 8, 1)),
        extracteur,  # Réutilise les motifs appris, sans les réentraîner.
        tf.keras.layers.Dense(1, activation="sigmoid"),  # Score de la seule classe « pair ».
    ])
    modele_parite.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])


In [ ]:
if executer_transfert:
    modele_parite.fit(
        X_train_cnn, y_train_parite,
        validation_data=(X_val_cnn, y_val_parite),
        epochs=6, batch_size=32, verbose=0,
    )  # Seule la nouvelle tête reçoit des corrections.
    loss_parite, accuracy_parite = modele_parite.evaluate(X_test_cnn, y_test_parite, verbose=0)
    print("Poids réentraînables de l'extracteur :", len(extracteur.trainable_weights))
    print("Accuracy pair / impair :", round(accuracy_parite, 3))
    print("0 poids réentraînables confirme le gel ; le score mesure seulement ce nouveau problème.")
